# NBA Scout Data Processing

Notebook này dùng để thử nghiệm data processing trước khi chuyển logic vào codebase local.

Mục tiêu tạo 3 gold datasets:

- `player_role_features.parquet`: role profile và similarity features.
- `performance_training.parquet`: rolling form features và future production targets.
- `salary_training.parquet`: contract/salary-cap-adjusted training table.

Notebook cố tình không import code local trong `src/` hoặc `app/` để giữ giai đoạn exploration tách biệt.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 1. Paths and Input Contracts

Đặt raw data vào `data/raw/`. Nếu tên file khác, sửa biến path bên dưới trong notebook này.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

PLAYERS_PATH = RAW_DIR / "players.parquet"
GAME_LOGS_PATH = RAW_DIR / "player_game_logs.parquet"
SEASON_STATS_PATH = RAW_DIR / "player_season_stats.parquet"
CONTRACTS_PATH = RAW_DIR / "contracts.parquet"
SALARY_CAP_PATH = RAW_DIR / "salary_cap_by_season.parquet"

OUTPUT_ROLE_FEATURES = GOLD_DIR / "player_role_features.parquet"
OUTPUT_PERFORMANCE_TRAINING = GOLD_DIR / "performance_training.parquet"
OUTPUT_SALARY_TRAINING = GOLD_DIR / "salary_training.parquet"

RAW_DIR, GOLD_DIR

In [ ]:
RAW_SCHEMAS = {
    "players": [
        "player_id", "player_name", "birth_date", "position", "height", "weight",
    ],
    "player_game_logs": [
        "player_id", "game_date", "game_id", "season", "team_id", "minutes", "points", "assists",
        "rebounds", "usage_pct", "true_shooting_pct", "opponent", "home_away", "rest_days",
    ],
    "player_season_stats": [
        "player_id", "season", "team_id", "age", "minutes", "usage_pct", "points_per_100",
        "assists_per_100", "rebounds_per_100", "true_shooting_pct", "three_point_attempt_rate",
        "free_throw_rate", "turnover_rate", "steal_rate", "block_rate", "offensive_rating",
        "defensive_rating",
    ],
    "contracts": [
        "player_id", "contract_signed_date", "season", "annual_salary", "contract_years", "previous_salary",
    ],
    "salary_cap_by_season": ["season", "salary_cap"],
}

def read_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")

def report_schema(name: str, df: pd.DataFrame, expected_columns: list[str]) -> None:
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    missing = sorted(set(expected_columns) - set(df.columns))
    extra = sorted(set(df.columns) - set(expected_columns))
    if missing:
        print("  Missing expected columns:", missing)
    if extra:
        print("  Extra columns:", extra[:25], "..." if len(extra) > 25 else "")

## 2. Load Raw Tables

In [ ]:
players = read_table(PLAYERS_PATH)
game_logs = read_table(GAME_LOGS_PATH)
season_stats = read_table(SEASON_STATS_PATH)
contracts = read_table(CONTRACTS_PATH)
salary_cap = read_table(SALARY_CAP_PATH)

tables = {
    "players": players,
    "player_game_logs": game_logs,
    "player_season_stats": season_stats,
    "contracts": contracts,
    "salary_cap_by_season": salary_cap,
}

for table_name, table_df in tables.items():
    report_schema(table_name, table_df, RAW_SCHEMAS[table_name])

## 3. Gold Dataset 1: Player Role Features

Một dòng = một cầu thủ trong một mùa. Dataset này phục vụ player similarity, candidate retrieval, role explanation, và có thể tái sử dụng cho salary model.

In [ ]:
ROLE_BASE_FEATURES = [
    "minutes",
    "usage_pct",
    "points_per_100",
    "assists_per_100",
    "rebounds_per_100",
    "true_shooting_pct",
    "three_point_attempt_rate",
    "free_throw_rate",
    "turnover_rate",
    "steal_rate",
    "block_rate",
    "offensive_rating",
    "defensive_rating",
]

def add_role_dimensions(df: pd.DataFrame) -> pd.DataFrame:
    role = df.copy()
    role["scoring_creation"] = role[["points_per_100", "usage_pct", "free_throw_rate"]].mean(axis=1)
    role["playmaking"] = role[["assists_per_100", "usage_pct"]].mean(axis=1) - role["turnover_rate"].fillna(0)
    role["shooting"] = role[["true_shooting_pct", "three_point_attempt_rate"]].mean(axis=1)
    role["rim_pressure"] = role[["free_throw_rate", "points_per_100"]].mean(axis=1)
    role["rebounding"] = role["rebounds_per_100"]
    role["perimeter_defense"] = role["steal_rate"]
    role["interior_defense"] = role["block_rate"]
    role["two_way_impact"] = role["offensive_rating"] - role["defensive_rating"]
    return role

ROLE_DIMENSIONS = [
    "scoring_creation", "playmaking", "shooting", "rim_pressure", "rebounding",
    "perimeter_defense", "interior_defense", "two_way_impact",
]

def build_player_role_features(season_stats_df: pd.DataFrame, players_df: pd.DataFrame) -> pd.DataFrame:
    if season_stats_df.empty:
        return pd.DataFrame()

    required = ["player_id", "season", "team_id", "age"] + ROLE_BASE_FEATURES
    role = season_stats_df[required].copy()
    role = add_role_dimensions(role)

    identity_cols = [col for col in ["player_id", "player_name", "position"] if col in players_df.columns]
    if identity_cols:
        role = role.merge(players_df[identity_cols].drop_duplicates("player_id"), on="player_id", how="left")

    output_cols = [
        "player_id", "player_name", "season", "team_id", "age", "position",
        *ROLE_BASE_FEATURES,
        *ROLE_DIMENSIONS,
    ]
    output_cols = [col for col in output_cols if col in role.columns]
    return role[output_cols].sort_values(["season", "player_id"]).reset_index(drop=True)

player_role_features = build_player_role_features(season_stats, players)
player_role_features.head()

In [ ]:
if not player_role_features.empty:
    player_role_features.to_parquet(OUTPUT_ROLE_FEATURES, index=False)
    print(f"Saved {OUTPUT_ROLE_FEATURES} with shape {player_role_features.shape}")

### Quick Similarity Prototype

Cell này giúp kiểm tra nhanh câu hỏi: cầu thủ nào có style/role gần một cầu thủ target. Có thể đổi `TARGET_PLAYER_NAME`, `TARGET_SEASON`, và `SIMILARITY_WEIGHTS`.

In [ ]:
TARGET_PLAYER_NAME = "LeBron James"
TARGET_SEASON = None

SIMILARITY_WEIGHTS = {
    "scoring_creation": 1.0,
    "playmaking": 1.2,
    "shooting": 0.8,
    "rim_pressure": 1.0,
    "rebounding": 0.7,
    "perimeter_defense": 0.8,
    "interior_defense": 0.4,
    "two_way_impact": 0.8,
}

def find_similar_players(
    role_df: pd.DataFrame,
    target_player_name: str,
    target_season: str | int | None = None,
    top_k: int = 10,
    weights: dict[str, float] | None = None,
) -> pd.DataFrame:
    if role_df.empty:
        return pd.DataFrame()

    features = [feature for feature in ROLE_DIMENSIONS if feature in role_df.columns]
    matrix = role_df[features].copy()
    if weights:
        for feature, weight in weights.items():
            if feature in matrix.columns:
                matrix[feature] = matrix[feature] * weight

    preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    X = preprocessor.fit_transform(matrix)

    target_mask = role_df["player_name"].str.lower().eq(target_player_name.lower())
    if target_season is not None:
        target_mask &= role_df["season"].eq(target_season)
    if not target_mask.any():
        raise ValueError(f"Target player not found: {target_player_name}, season={target_season}")

    target_idx = role_df[target_mask].index[-1]
    sims = cosine_similarity(X[target_idx : target_idx + 1], X).ravel()

    results = role_df.copy()
    results["similarity_score"] = sims
    results = results.loc[results.index != target_idx]
    return results.sort_values("similarity_score", ascending=False).head(top_k)

if not player_role_features.empty and "player_name" in player_role_features.columns:
    similar_players = find_similar_players(
        player_role_features,
        TARGET_PLAYER_NAME,
        target_season=TARGET_SEASON,
        top_k=10,
        weights=SIMILARITY_WEIGHTS,
    )
    display(similar_players[["player_name", "season", "team_id", "position", "similarity_score", *ROLE_DIMENSIONS]])

## 4. Gold Dataset 2: Performance Training

Một dòng = một cầu thủ tại một `as_of_date`. Features chỉ dùng dữ liệu trước hoặc tại `as_of_date`; targets dùng trung bình 5 trận tiếp theo.

In [ ]:
def add_rolling_player_features(game_logs_df: pd.DataFrame) -> pd.DataFrame:
    if game_logs_df.empty:
        return pd.DataFrame()

    df = game_logs_df.copy()
    df["game_date"] = pd.to_datetime(df["game_date"])
    df = df.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    grouped = df.groupby("player_id", group_keys=False)

    stat_prefixes = {"points": "pts", "assists": "ast", "rebounds": "reb"}

    for stat, prefix in stat_prefixes.items():
        for window in [5, 10, 20]:
            df[f"{prefix}_last_{window}"] = grouped[stat].transform(
                lambda s: s.shift(1).rolling(window=window, min_periods=1).mean()
            )
        df[f"{prefix}_season_to_date"] = grouped[stat].transform(
            lambda s: s.shift(1).expanding(min_periods=1).mean()
        )

    for stat in ["minutes", "usage_pct", "true_shooting_pct"]:
        df[f"{stat}_last_5"] = grouped[stat].transform(
            lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
        )
        df[f"{stat}_last_10"] = grouped[stat].transform(
            lambda s: s.shift(1).rolling(window=10, min_periods=1).mean()
        )
        df[f"{stat}_trend"] = df[f"{stat}_last_5"] - df[f"{stat}_last_10"]

    for stat in ["points", "assists", "rebounds"]:
        df[f"target_next_5_games_{stat}"] = grouped[stat].transform(
            lambda s: s.shift(-1).rolling(window=5, min_periods=1).mean().shift(-4)
        )

    df = df.rename(columns={"game_date": "as_of_date"})
    return df

def build_performance_training(game_logs_df: pd.DataFrame) -> pd.DataFrame:
    features = add_rolling_player_features(game_logs_df)
    if features.empty:
        return pd.DataFrame()

    id_cols = [
        "player_id", "as_of_date", "game_id", "season", "team_id", "opponent", "home_away", "rest_days",
    ]
    feature_cols = [
        "pts_last_5", "pts_last_10", "pts_last_20", "pts_season_to_date",
        "ast_last_5", "ast_last_10", "ast_last_20", "ast_season_to_date",
        "reb_last_5", "reb_last_10", "reb_last_20", "reb_season_to_date",
        "minutes_last_5", "minutes_last_10", "minutes_trend",
        "usage_pct_last_5", "usage_pct_last_10", "usage_pct_trend",
        "true_shooting_pct_last_5", "true_shooting_pct_last_10", "true_shooting_pct_trend",
    ]
    target_cols = [
        "target_next_5_games_points", "target_next_5_games_assists", "target_next_5_games_rebounds",
    ]
    output_cols = [col for col in [*id_cols, *feature_cols, *target_cols] if col in features.columns]
    return features[output_cols].dropna(subset=target_cols, how="all").reset_index(drop=True)

performance_training = build_performance_training(game_logs)
performance_training.head()

In [ ]:
if not performance_training.empty:
    performance_training.to_parquet(OUTPUT_PERFORMANCE_TRAINING, index=False)
    print(f"Saved {OUTPUT_PERFORMANCE_TRAINING} with shape {performance_training.shape}")

## 5. Gold Dataset 3: Salary Training

Một dòng = một lần ký hợp đồng. Target chính nên là `target_salary_cap_percentage` để so sánh công bằng giữa các mùa.

In [ ]:
def build_salary_training(
    contracts_df: pd.DataFrame,
    salary_cap_df: pd.DataFrame,
    role_features_df: pd.DataFrame,
    players_df: pd.DataFrame,
) -> pd.DataFrame:
    if contracts_df.empty or salary_cap_df.empty:
        return pd.DataFrame()

    salary = contracts_df.copy()
    salary["contract_signed_date"] = pd.to_datetime(salary["contract_signed_date"])
    salary = salary.merge(salary_cap_df[["season", "salary_cap"]], on="season", how="left")
    salary["target_annual_salary"] = salary["annual_salary"]
    salary["target_salary_cap_percentage"] = salary["annual_salary"] / salary["salary_cap"]
    salary["target_contract_years"] = salary["contract_years"]

    if not role_features_df.empty:
        role_for_salary = role_features_df.copy()
        role_for_salary["next_season"] = role_for_salary["season"] + 1
        keep_cols = ["player_id", "next_season", "age", "position", *ROLE_BASE_FEATURES, *ROLE_DIMENSIONS]
        keep_cols = [col for col in keep_cols if col in role_for_salary.columns]
        salary = salary.merge(
            role_for_salary[keep_cols],
            left_on=["player_id", "season"],
            right_on=["player_id", "next_season"],
            how="left",
        )

    if "birth_date" in players_df.columns:
        birth_dates = players_df[["player_id", "birth_date"]].copy()
        birth_dates["birth_date"] = pd.to_datetime(birth_dates["birth_date"])
        salary = salary.merge(birth_dates, on="player_id", how="left")
        salary["age_at_signing"] = (salary["contract_signed_date"] - salary["birth_date"]).dt.days / 365.25
    elif "age" in salary.columns:
        salary["age_at_signing"] = salary["age"]

    output_cols = [
        "player_id", "contract_signed_date", "season", "age_at_signing", "position",
        "salary_cap", "previous_salary", "target_annual_salary", "target_salary_cap_percentage",
        "target_contract_years", *ROLE_BASE_FEATURES, *ROLE_DIMENSIONS,
    ]
    output_cols = [col for col in output_cols if col in salary.columns]
    return salary[output_cols].sort_values(["contract_signed_date", "player_id"]).reset_index(drop=True)

salary_training = build_salary_training(contracts, salary_cap, player_role_features, players)
salary_training.head()

In [ ]:
if not salary_training.empty:
    salary_training.to_parquet(OUTPUT_SALARY_TRAINING, index=False)
    print(f"Saved {OUTPUT_SALARY_TRAINING} with shape {salary_training.shape}")

## 6. Basic Data Quality Checks

Các check này chỉ để exploration. Khi logic ổn, có thể chuyển thành test hoặc Great Expectations suite sau.

In [ ]:
def quality_summary(name: str, df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        print(f"{name}: empty")
        return pd.DataFrame()
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean(),
        "n_unique": df.nunique(dropna=True),
    })
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    return summary.sort_values("missing_pct", ascending=False)

quality_summary("player_role_features", player_role_features).head(20)

In [ ]:
quality_summary("performance_training", performance_training).head(20)

In [ ]:
quality_summary("salary_training", salary_training).head(20)

## 7. Next Decisions

- Chốt raw data source thực tế cho `players`, `player_game_logs`, `player_season_stats`, `contracts`, `salary_cap_by_season`.
- Chốt định nghĩa `season` và mapping mùa ký hợp đồng sang stats mùa trước.
- Chốt role dimensions: công thức hiện tại chỉ là baseline heuristic để exploration, chưa phải product formula cuối.
- Chốt target horizon cho performance forecast: 5 games, 10 games, hoặc remainder-of-season.
- Sau khi notebook chạy ổn, chuyển từng function thành module xử lý dữ liệu có test.